vielä kerran: henkilöittäin, 2015-2025, varianssianalyysi per hlö
- ympätään datat 2015-2025
- erotellaan henkilöittäin
- pidetään puheiden aikaleimat, paino pois vuosilta
- lemmatisoitu ja lemmatisoimaton
- varianssi ajan myötä, tarkastelu per henkilö
- koitetaan saada ennen ennen kesälomia

In [1]:
# import packages, declare constants, get the parent directory
import pandas as pd
import numpy as np
import re
import os
import time
#import decimal
from scipy import stats
#from wordcloud import WordCloud
import matplotlib.pyplot as plt
from matplotlib import colormaps
#from .utils import helpers
#import simplemma
import pyvoikko

CHATGPT_RELEASE_YEAR = int(2022)
FINNISH_ALPHABET = 'abcdefghijklmnopqrstuvwxyzåäö'

def get_parent_directory() -> str:
    """Get the parent directory for handling csv files.

    Returns:
        string: the path to the directory where directories for csv files are located
    """
    #create relative path for parent
    relative_parent = os.path.join(os.getcwd(), '..')

    #use abspath for absolute parent path
    return str(os.path.abspath(relative_parent)).replace('\\', '/')

directory = get_parent_directory()

In [ ]:
# Read the csvs of speeches, years 2015-2025. Combine into a singel dataframe. The dataframe will include a column with lemmatised version of the speech 
# and an original unlemmatised version.
# Running this cell does nothing but creates the dataset for analysis.
#for csv in [i for i in os.listdir(f'{directory}/csv_lemmatized/') if re.search('\d+',i)[0] >= 2015]:
df_all_years = pd.DataFrame()

for csv in [i for i in os.listdir(f'{directory}/csv_rawdata/') if re.search(r'\d+', i)]:
    if int(re.search(r'\d+', csv)[0]) < 2015:
        pass
    else:
        if int(re.search(r'\d+', csv)[0]) < 2025:
            df_year = pd.read_csv(f'{directory}/csv_rawdata/{csv}', sep=',', header=0, encoding='utf-8', on_bad_lines='skip') # OBS OBS OBS: on_bad_lines='skip' !!!!!!!!!!!!!!
            df_all_years = pd.concat([df_all_years, df_year])
        else:
            df_year = pd.read_csv(f'{directory}/csv_rawdata/{csv}', sep=';', header=0, encoding='utf-8', on_bad_lines='skip') # OBS OBS OBS: on_bad_lines='skip' !!!!!!!!!!!!!!
            df_all_years = pd.concat([df_all_years, df_year]) 

In [23]:
# Keep only rows where the speaker identifier (speaker_id) is present. Some are missing for some reason.
df_all_years = df_all_years.loc[pd.notna(df_all_years['speaker_id'])]
df_all_years = df_all_years.loc[df_all_years['speaker_id'].str.strip().str.len()>0]

In [24]:
# Keep only rows with the speech_type category in: 'Esittelypuheenvuoro', 'Ryhmäpuheenvuoro', 'Varsinainen puheenvuoro', 'Puheenvuoro'
#    -> Drop rows with other categories.
#    This is to limit the rows to speeches where preparation (ergo option for using generative AI) is necessary.
# Clean and prep data before filtering:
# - Clean speech_type, fix known spelling errors

list_wanted_speech_types = ['Esittelypuheenvuoro', 'Ryhmäpuheenvuoro', 'Varsinainen puheenvuoro', 'Puheenvuoro']

def clean_speech_type(speech_type: str) -> str:
    if 'vastauspuheenvuoro' in speech_type:
        return 'Vastauspuheenvuoro'
    elif 'esittelypuheenvuoro' in speech_type:
        return 'Esittelypuheenvuoro'
    else:
        return speech_type.strip()

# Clean speech_type
df_all_years.loc[:, 'speech_type'] = df_all_years['speech_type'].apply(clean_speech_type)
# Filter rows
df_all_years = df_all_years[df_all_years['speech_type'].isin(list_wanted_speech_types)]

In [25]:
# Keep only rows where content is not None and the length of content is over 0 
# -> actual information is stored in there
df_all_years = df_all_years.loc[(pd.notna(df_all_years['content']))&(len(df_all_years['content'])>0)]

In [26]:
# Clean content (content column). Drop special characters etc., so that the content is similar throughout the dataframe
# and special characters etc. do not interfere.
# A small wrapper for the clean_string function for this purpose.
def clean_string(string: str) -> str:
    try:
        # remove blanks in start and end
        string = string.strip()
        string = string.lower()
        # the string must contain characters
        if any(c in string for c in FINNISH_ALPHABET)==False:
            string = ''
        # remove tabulations, line breaks etc., also special characters
        remove_these = r'[\+\*!"”’?.:,…()§\'[\] \t\n\r\f\v]'
        string = re.sub(remove_these, '', string)
        # remove weird parentheses and backwards linebreaks from starts of strings
        string = re.sub(r'^\)\\[a-z]', '', string)
        # remove weird '\[alphabet]' strings at start of strings
        string = re.sub(r'^\\[a-z]', '', string)
        # remove numbers
        string = re.sub(r'[0-9]', '', string)
        # remove dashes '-' at the start and end of string
        string = re.sub(r'^-|-$', '', string)
        # remove individual forward and backward slashes '/', '\'
        string = re.sub(r'[\/\\]', '', string)
        # remove double dashes '--'
        string = string.replace('--', '-')
        # remove the equal sign '='
        string = string.replace('=', '')
        # at the end of the cleaning, remove all characters from the string which are not in the alphabet except for dash (compound words)
        remove_these = ''.join([str(c) for c in string if c != '-' and c not in [i for i in FINNISH_ALPHABET]])
        string = re.sub(remove_these, '', string)
        # remove blanks in start and end again
        string = string.strip()
        # remove empty if string length < 2
        string = '' if len(string) < 2 else string
        return string
    except:
        print(f'Unexpected error at helpers.clean_string(), string: {string}')
        raise

def clean_string_wrapper(str_in: str) -> str:
    # format output string
    str_out = str()
    # turn input string into a list of words, cut at SPACE
    str_in = str_in.split()
    for i in str_in:
        i = clean_string(i)
        if (i is not None) and (len(i)>0):
            str_out = str_out+' '+i
    if len(str_out)==0:
        return None
    else:
        return str_out.strip()

df_all_years.loc[:, 'content'] = df_all_years.apply(lambda x: clean_string_wrapper(x['content']), axis=1)

# If lemmatisation and string cleaning has nulled any speeches, drop such rows
df_all_years = df_all_years.loc[pd.notna(df_all_years['content'])]
df_all_years.reset_index(inplace=True)

In [29]:
# Data preparation is now finished.
# What the dataframe looks like now:
df_all_years[0:1].T

,0
index,24
speech_id,2015_2_6
session,2/2015
date,2015-05-05
start_time,14.01
end_time,14.15
given,Sirkka-Liisa
family,Anttila
role,Kansanedustaja
party,KESK


The analysis part starts here.
Next the goal is to calculate word frequencies per speaker to see if there are significant changes speaker-wise.
The data will not be grouped by year, so the frequencies will not be based on yearly aggregates of observations. Frequencies will be calculated for each speech. Each speech has a unique speech identifier (speech_id).
To make it possible to do this for each speaker, several different datasets will be created:
- A list that holds all unique speaker identifiers.
- A dataframe that holds speaker_id, speech_id and session identifier (session)
- A dataframe that holds all words and their frequencies per speech

In [30]:
# Next:
# - Store each speaker_id (speaker identifier) in a list: speakers
speakers = df_all_years['speaker_id'].unique()

In [31]:
# Next: Get each speakers' speech and session identifiers. Store them in a dataframe: df_speeches_sessions
df_speeches_sessions = pd.DataFrame(columns=['speaker_id','session','speech_id','content']).astype({'speaker_id':str, 'session':str, 'speech_id':str, 'content':str})

# Populate the dataframe
for speaker in speakers:
    df_speeches_sessions = pd.concat([df_speeches_sessions, df_all_years[['speaker_id','session','speech_id','content']].loc[df_all_years['speaker_id']==speaker]], axis=0, ignore_index=True)

In [42]:
# Next: Get all words per speaker per speech. Store them in a dataframe: df_speech_session_words.
# Store the dataframe (containing information 'speaker_id','session','speech_id','word','word_n') as a csv:
#    a csv will be created for each speaker; each speaker's data will be in their respective csv.
def count_word_freqs_in_string(string: str):
    """Counts the words in the input string.
    Returns a dictionary where the word is the key and the frequency is the value.
    """
    if ((string is None) or (string == 'nan')):
        return None
    else:
        words_list = re.split(' ', string)
        wordfreq_dict = {}
        for word in words_list:
            if word not in wordfreq_dict.keys():
                wordfreq_dict[word] = 1
            else:
                wordfreq_dict[word] += 1

        return wordfreq_dict

# Populate the dataframe with help from the function declared above
for speaker in speakers:
    # A name for csv in a helper variable.
    csv_name = f'speaker_{speaker}_words_speeches_sessions.csv'
    csv_save_directory = f'{directory}/variance_person/csv_nonlemmatised/speaker_words_sessions'
    csv_save_path = f'{csv_save_directory}/{csv_name}'
    # If the csv already exists (speaker's data has already been created) -> skip and move to the next speaker
    if csv_name in os.listdir(csv_save_directory):
        pass
    else:
        df_filtered_by_speaker = df_speeches_sessions[df_speeches_sessions['speaker_id']==speaker]
        df_speech_session_words = df_filtered_by_speaker.assign(word=df_speeches_sessions['content'].str.split()).explode('word').groupby(['speaker_id','session','speech_id','word']).size().reset_index(name='word_n')
        df_speech_session_words = df_speech_session_words[['speaker_id','session','speech_id','word','word_n']]
        # Store the speaker's dataframe as a csv
        try:
            df_speech_session_words.to_csv(csv_save_path, sep=';', header=True, index=False, encoding='utf-8')
        except FileExistsError:
            os.remove(csv_save_path)
            df_speech_session_words.to_csv(csv_save_path, sep=';', header=True, index=False, encoding='utf-8')

Data preparation at the individual (speaker) level is now finished.

In [45]:
# Simple analysis of word frequency development per person over SESSIONS.
# Analysis done per each speaker and for each of their sessions. Calculating word frequencies in the sessions, in order to find significant changes over time (sessions).
# > Select person 
# -> Get person's sessions 
# -> For each session, calculate word frequency for each word 
# -> For each session, calculate z-score for each word 
# -> For each word, see if significant changes take place over time (session).

# each file has the same structure: speaker_id, session, speech_id, word, word_n
datatypes = {'speaker_id':str, 'session':str, 'speech_id':str, 'word':str, 'word_n':int}

for csv in [i for i in os.listdir(f'{directory}/variance_person/csv_nonlemmatised/speaker_words_sessions/') if re.search(r'speaker_\d+_words_speeches_sessions', i)]:
    file_path = f'{directory}/variance_person/csv_nonlemmatised/speaker_words_sessions/{csv}'
    df = pd.read_csv(file_path, sep=';', header=0, dtype=datatypes)
    speaker = df['speaker_id'].unique() # store speaker_id, there is only 1 unique value as each csv is speaker specific
    sessions_list = df['session'].unique().astype(str)
    # create a dataframe to store all speaker specific results, ie data from all sessions created below
    combine_frequencies_df = pd.DataFrame(columns=['speaker_id','session','word','word_n','word_freq','word_z']).astype({'speaker_id':str,'session':str,'word':str,'word_n':int,'word_freq':float,'word_z':float})
    for session in sessions_list:
        # begin calculating frequencies and z-score: 
        # extract the session specific rows
        session_df = df[df['session']==session]
        # create a dataframe for calculating the z-score: word, word_n, session, word_freq, word_z
        # -> calculate the sum of word_n's for cases where the speaker could have several speeches in the same session
        calc_df = session_df.groupby('word', as_index=False)['word_n'].sum()
        # -> calculate session specific frequency for each word
        calc_df['word_freq'] = (calc_df['word_n']/calc_df['word_n'].sum())
        # -> calculate session specific z-score for each word
        calc_df['word_z'] = calc_df['word_n'].transform(lambda s: (s - s.mean()) / s.std())
        # add speaker and session information to ease storing into combine_frequencies_df
        calc_df['session'] = session #session[0]
        calc_df['speaker_id'] = speaker[0]
        # store the data in combine df
        combine_frequencies_df = pd.concat([combine_frequencies_df, calc_df[['speaker_id','session','word','word_n','word_freq','word_z']]])
    # now that the the data has been created, store again as csv to make life easier
    csv_name = f'speaker_{speaker[0]}_words_frequencies_sessions.csv'
    csv_save_directory = f'{directory}/variance_person/csv_nonlemmatised/session_frequency_z_score'
    csv_save_path = f'{csv_save_directory}/{csv_name}'
    # If the csv already exists (speaker's data has already been created) -> skip and move to the next speaker
    if csv_name in os.listdir(csv_save_directory):
        pass
    else:
        # Store the speaker's dataframe as a csv
        try:
            combine_frequencies_df.to_csv(csv_save_path, sep=';', header=True, index=False, encoding='utf-8')
        except FileExistsError:
            os.remove(csv_save_path)
            combine_frequencies_df.to_csv(csv_save_path, sep=';', header=True, index=False, encoding='utf-8')

In [2]:
# Now look for significant increases in z-scores and/or frequencies.
# Limit analysis to speakers who appear in sessions before and after 2022.

# OBS OBS OBS
# this has now only looked at INCREASES after a cutoff point - how about decreases around the release of ChatGPT?

# each file has the same structure: speaker_id, session, word, word_n, word_freq, word_z
datatypes = {'speaker_id':str, 'session':str, 'word':str, 'word_n':int, 'word_freq':float, 'word_z':float}

# format a dataframe to store information about the speakers' words' z-values/frequencies before and after CHATGPT_RELEASE_YEAR
speakers_z_info = pd.DataFrame(columns=['speaker_id','word','z_before','z_after','z_diff']).astype({'speaker_id':str,'word':str,'z_before':float,'z_after':float,'z_diff':float})
speakers_freq_info = pd.DataFrame(columns=['speaker_id','word','freq_before','freq_after','freq_diff']).astype({'speaker_id':str,'word':str,'freq_before':float,'freq_after':float,'freq_diff':float})

# a path for saving plots drawn
fig_save_path = f'{directory}/variance_person/plots'

for csv in [i for i in os.listdir(f'{directory}/variance_person/csv_nonlemmatised/session_frequency_z_score/') if re.search(r'speaker_\d+_words_frequencies_sessions', i)]:
    file_path = f'{directory}/variance_person/csv_nonlemmatised/session_frequency_z_score/{csv}'
    df = pd.read_csv(file_path, sep=';', header=0, dtype=datatypes)
    # check if sessions from years before 2023 and after 2022 exist
    if (len([i for i in df['session'].unique() if int(i[-4:]) < CHATGPT_RELEASE_YEAR])>0 and len([i for i in df['session'].unique() if int(i[-4:]) > CHATGPT_RELEASE_YEAR])>0):
        # yes: continue
        # get speaker id
        speaker = df['speaker_id'][0]
        # get words that also exist in sessions before and after CHATGPT_RELEASE_YEAR (2022)
        words_list = [word for word in df['word'].unique() if (word in df['word'][df['session'].isin([session for session in df['session'].unique() if int(session[-4:])<CHATGPT_RELEASE_YEAR])].tolist()) & (word in df['word'][df['session'].isin([session for session in df['session'].unique() if int(session[-4:])>CHATGPT_RELEASE_YEAR])].tolist())]
        # extract the rows regarding these words into a new dataframe: words_df
        words_df = df[df['word'].isin(words_list)]

        # set x-axis for plot: sessions
        sessions_list = words_df['session'].unique().tolist()
        # sort the list: sessions and list into chronological order
        sessions_list = sorted(sessions_list, key=lambda x: (int(x.split('/')[1]), int(x.split('/')[0])))

        # naive search for words whose z-score is higher in years after the release of ChatGPT than before
        # index of cutoff in sessions_list: this shall be the first session after the year 2022
        cutoff = [i for i in sessions_list if int(i[-4:]) > CHATGPT_RELEASE_YEAR][0]
        cutoff_idx = sessions_list.index(cutoff)
        # store the sessions before and after the cutoff point to lists
        sessions_before = sessions_list[:cutoff_idx]
        sessions_after = sessions_list[cutoff_idx:]

        # pivot words_df to wide format: words as rows, sessions as columns
        # -> words are the index row, each session its own column, and if a word was not used in that session its z-value/frequency will be NaN
        words_df_z_pivot = words_df.pivot(index="word", columns="session", values="word_z")
        words_df_freq_pivot = words_df.pivot(index="word", columns="session", values="word_freq")
        
        # NOTE the columns (sessions) in the words_df_z_pivot/words_df_freq_pivot dataframe are NOT IN SORTED ORDER
        # -> to create the plot correctly, we have to sort the data
        # -> sort with sessions_list since the columns are the sessions
        words_df_z_pivot = words_df_z_pivot[sessions_list]
        words_df_freq_pivot = words_df_freq_pivot[sessions_list]

        # get the mean of z-scores before and after cutoff point
        z_mean_before = words_df_z_pivot.iloc[:, :cutoff_idx].mean(axis=1)
        z_mean_after = words_df_z_pivot.iloc[:, cutoff_idx:].mean(axis=1)
        # get the difference of z-scores before and after the cutoff
        z_mean_diff = z_mean_after-z_mean_before

        freq_mean_before = words_df_freq_pivot.iloc[:, :cutoff_idx].mean(axis=1)
        freq_mean_after = words_df_freq_pivot.iloc[:, cutoff_idx:].mean(axis=1)
        freq_mean_diff = freq_mean_after-freq_mean_before

        # get the top N largest differences from z_mean_diff/freq_mean_diff dataframe's index, where the words make the index
        # -> store the words in a list
        # -> create the plot from these words
        words_z_means_largest_diffs = z_mean_diff.nlargest(20).index.tolist()
        words_freq_means_largest_diffs = freq_mean_diff.nlargest(20).index.tolist()

        # draw plots
        # format a figure
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))
        fig.suptitle(f'{speaker} (nonlemmatised text)', y=1.0, fontsize=14)
        # sessions to numeric indices - x-axis
        x_axis = np.arange(len(sessions_list))
        
        # PLOT 1: Z-SCORES
        # playing and sugarcoating with colours
        colours = plt.cm.tab20(np.linspace(0, 1, len(words_z_means_largest_diffs)))
        # plot each word, but sort the data 
        for word, colour in zip(words_z_means_largest_diffs, colours):
            # NOTE this draws a line between all plot, even non-consecutive observations!
            y = words_df_z_pivot.loc[word].values
            # filter out NaN values (keep only valid sessions for this word)
            valid_mask = ~np.isnan(y)
            ax1.plot(
                # numeric x-axis
                x_axis[valid_mask],
                # a map of missing and existing values
                y[valid_mask],
                label=word,
                linewidth=1,
                linestyle="-",
                marker="o",
                markersize=2,
                color=colour
            )
        # add vertical cutoff line
        ax1.axvline(x=x_axis[cutoff_idx], color="red", linestyle="--", label="CUTOFF")
        ax1.set_title('Word (nonlemmatised) z-score over sessions')
        ax1.set_xticks(x_axis)
        ax1.set_xticklabels(sessions_list, rotation=60, fontsize=6)
        ax1.set_xlabel("Session")
        ax1.set_ylabel("Z-score")
        # legend as a table below the plot
        ax1.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, -0.2),  # (x, y) position below the plot
            ncol=5,                      
            fontsize=7,                  
            frameon=False       
        )
        ax1.grid(True)

        # PLOT 2: FREQUENCIES
        # playing and sugarcoating with colours
        colours = plt.cm.tab20(np.linspace(0, 1, len(words_freq_means_largest_diffs)))
        # plot each word, but sort the data 
        for word, colour in zip(words_freq_means_largest_diffs, colours):
            # NOTE this draws a line between all plot, even non-consecutive observations!
            y = words_df_z_pivot.loc[word].values
            # filter out NaN values (keep only valid sessions for this word)
            valid_mask = ~np.isnan(y)
            ax2.plot(
                # numeric x-axis
                x_axis[valid_mask],
                # a map of missing and existing values
                y[valid_mask],
                label=word,
                linewidth=1,
                linestyle="-",
                marker="o",
                markersize=2,
                color=colour
            )
        # add vertical cutoff line
        ax2.axvline(x=x_axis[cutoff_idx], color="red", linestyle="--", label="CUTOFF")
        ax2.set_title('Word (nonlemmatised) frequency over sessions')
        ax2.set_xticks(x_axis)
        ax2.set_xticklabels(sessions_list, rotation=60, fontsize=6)        
        ax2.set_xlabel("Session")
        ax2.set_ylabel("Frequency")
        # legend as a table below the plot
        ax2.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, -0.2),  # (x, y) position below the plot
            ncol=5,                      
            fontsize=7,                  
            frameon=False       
        )
        ax2.grid(True)        
        
        # sugar frosting: adjust top margin to fit legend
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        # save as a file
        fig_save_file_name = f'fig_{speaker}_sessions_z_freq_nonlemmatised.png'  
        plt.savefig(f'{fig_save_path}/{fig_save_file_name}', dpi=300, bbox_inches="tight") 
        plt.close(fig=None)

        # save information of the most interesting words in a dataframe with all speakers
        z_mean_before = z_mean_before.rename('z_before')
        z_mean_after = z_mean_after.rename('z_after')
        z_mean_diff = z_mean_diff.rename('z_diff')
        speaker_z_info_insert = pd.merge(z_mean_before, z_mean_after, on='word', how='left')
        speaker_z_info_insert = pd.merge(speaker_z_info_insert, z_mean_diff, on='word', how='left')
        speaker_z_info_insert = speaker_z_info_insert.reset_index(names='word')
        speaker_z_info_insert['speaker_id'] = speaker
        speaker_z_info_insert = speaker_z_info_insert[['speaker_id','word','z_before','z_after','z_diff']]
        speakers_z_info = pd.concat([speakers_z_info, speaker_z_info_insert], axis=0, ignore_index=True)

        freq_mean_before = freq_mean_before.rename('freq_before')
        freq_mean_after = freq_mean_after.rename('freq_after')
        freq_mean_diff = freq_mean_diff.rename('freq_diff')
        speaker_freq_info_insert = pd.merge(freq_mean_before, freq_mean_after, on='word', how='left')
        speaker_freq_info_insert = pd.merge(speaker_freq_info_insert, freq_mean_diff, on='word', how='left')
        speaker_freq_info_insert = speaker_freq_info_insert.reset_index(names='word')
        speaker_freq_info_insert['speaker_id'] = speaker
        speaker_freq_info_insert = speaker_freq_info_insert[['speaker_id','word','freq_before','freq_after','freq_diff']]
        speakers_freq_info = pd.concat([speakers_freq_info, speaker_freq_info_insert], axis=0, ignore_index=True)

    # if sessions from the wanted years do not exist in csv, do nothing
    else:
        pass

# Store the speakers' z-value information dataframe as a csv
try:
    speakers_z_info.to_csv(f'{directory}/variance_person/csv_nonlemmatised/session_frequency_z_score/all_speakers_z_scores_before_after_lemma.csv', sep=';', header=True, index=False, encoding='utf-8')
except FileExistsError:
    os.remove(f'{directory}/variance_person/csv_nonlemmatised/session_frequency_z_score/all_speakers_z_scores_before_after_lemma.csv')
    speakers_z_info.to_csv(f'{directory}/variance_person/csv_nonlemmatised/session_frequency_z_score/all_speakers_z_scores_before_after_lemma.csv', sep=';', header=True, index=False, encoding='utf-8')

# Store the speakers' freq information dataframe as a csv
try:
    speakers_freq_info.to_csv(f'{directory}/variance_person/csv_nonlemmatised/session_frequency_z_score/all_speakers_freq_before_after_lemma.csv', sep=';', header=True, index=False, encoding='utf-8')
except FileExistsError:
    os.remove(f'{directory}/variance_person/csv_nonlemmatised/session_frequency_z_score/all_speakers_freq_before_after_lemma.csv')
    speakers_freq_info.to_csv(f'{directory}/variance_person/csv_nonlemmatised/session_frequency_z_score/all_speakers_freq_before_after_lemma.csv', sep=';', header=True, index=False, encoding='utf-8')

In [ ]:
# Figures have been created and stored as well as information about z-values and frequencies before and after the cutoff point of words that appear before and after
# the cutoff point.
# Let's check if any words are shared between the speakers.
z_df = pd.read_csv(f'{directory}/variance_person/csv_nonlemmatised/session_frequency_z_score/all_speakers_z_scores_before_after_lemma.csv', sep=';', header=0, encoding='utf-8')
freq_df = pd.read_csv(f'{directory}/variance_person/csv_nonlemmatised/session_frequency_z_score/all_speakers_freq_before_after_lemma.csv', sep=';', header=0, encoding='utf-8')

# count by how many speakers each word is used
z_word_speaker_count = z_df.groupby('word', as_index=False)['speaker_id'].count()
freq_word_speaker_count = freq_df.groupby('word', as_index=False)['speaker_id'].count()

# filter dataframes: keep only rows where the word is one shared by more than one speaker -> filter out rows where the word is unique to the speaker
z_df_filter = z_df[z_df['word'].isin(z_word_speaker_count['word'][z_word_speaker_count['speaker_id']>1])]
freq_df_filter = freq_df[freq_df['word'].isin(freq_word_speaker_count['word'][freq_word_speaker_count['speaker_id']>1])]

In [20]:
z_word_speaker_count.sort_values('speaker_id', ascending=False).nlargest(n=50, columns='speaker_id')['word'].tolist()
#z_word_speaker_count.nlargest(n=50, columns='speaker_id')['word'].tolist()

['ovat',
 'se',
 'sen',
 'on',
 'että',
 'ja',
 'tämä',
 'sitä',
 'nyt',
 'mutta',
 'puhemies',
 'kun',
 'voi',
 'myös',
 'arvoisa',
 'jo',
 'jotka',
 'meidän',
 'siitä',
 'kuin',
 'ollut',
 'tässä',
 'tämän',
 'hyvä',
 'niin',
 'ei',
 'olisi',
 'vielä',
 'hallitus',
 'tästä',
 'niitä',
 'tulee',
 'tai',
 'kuitenkin',
 'joka',
 'täällä',
 'tähän',
 'jos',
 'esimerkiksi',
 'yksi',
 'ne',
 'vaan',
 'sitten',
 'ole',
 'siihen',
 'hallituksen',
 'mitä',
 'aikana',
 'sekä',
 'tätä']

In [19]:
#freq_df_filter[freq_df_filter['word'].isin(freq_word_speaker_count.nlargest(n=20, columns='speaker_id')['word'].tolist())]
freq_word_speaker_count.nlargest(n=50, columns='speaker_id')['word'].tolist()

['että',
 'ja',
 'kun',
 'mutta',
 'nyt',
 'on',
 'ovat',
 'puhemies',
 'se',
 'sen',
 'sitä',
 'tämä',
 'arvoisa',
 'ei',
 'hyvä',
 'jo',
 'jotka',
 'kuin',
 'meidän',
 'myös',
 'niin',
 'ollut',
 'siitä',
 'tämän',
 'tässä',
 'voi',
 'hallitus',
 'joka',
 'jos',
 'kuitenkin',
 'niitä',
 'olisi',
 'tai',
 'tulee',
 'tähän',
 'tästä',
 'täällä',
 'vielä',
 'aikana',
 'esimerkiksi',
 'hallituksen',
 'mitä',
 'ne',
 'ole',
 'pitää',
 'sekä',
 'siihen',
 'sitten',
 'tätä',
 'vaan']